# Module 17 — Decorators and Caching

**Part IV: Advanced** · Level: Advanced · Prerequisites: Modules 8, 10

## What you'll learn

- How decorators wrap functions
- Preserving metadata with `functools.wraps`
- Decorators that take arguments
- Memoization by hand and with `functools.lru_cache`
- `functools.partial`

---

## 17.1 What a decorator is

A **decorator** is a function that takes a function and returns a new (usually wrapped) function. It relies on three ideas you already know: functions are objects (Module 8), closures remember variables (Module 10), and `*args` / `**kwargs` forward any arguments.

In [ ]:
def log_calls(func):
    def wrapper(*args, **kwargs):
        print("calling", func.__name__)
        return func(*args, **kwargs)
    return wrapper

@log_calls
def add(a, b):
    return a + b

print(add(2, 3))

The `@log_calls` line is just shorthand for:

In [ ]:
def add2(a, b):
    return a + b
add2 = log_calls(add2)

## 17.2 Preserving metadata with `functools.wraps`

A bare wrapper replaces the original's `__name__` and docstring. `@wraps` copies them across.

In [ ]:
import functools

def broken(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

def good(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@broken
def hello():
    "Say hi."
    return "hi"

@good
def hello2():
    "Say hi."
    return "hi"

print(hello.__name__, "|", hello2.__name__, "|", hello2.__doc__)

Always use `@wraps` in decorators you write.

## 17.3 A practical decorator: timing

In [ ]:
import functools, time

def timed(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            elapsed = time.perf_counter() - start
            print(f"{func.__name__} took {elapsed:.4f}s")
    return wrapper

@timed
def slow_sum(n):
    return sum(range(n))

print(slow_sum(1_000_000))

## 17.4 Decorators with arguments

To configure a decorator, add one more level: a function that *returns* the decorator.

In [ ]:
import functools

def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(times=3)
def greet(name):
    print(f"hi {name}")

greet("Ada")

## 17.5 Stacking decorators

Decorators apply from the bottom up (closest to the function first).

In [ ]:
import functools

def bold(f):
    @functools.wraps(f)
    def w(*a, **k): return f"<b>{f(*a, **k)}</b>"
    return w

def italic(f):
    @functools.wraps(f)
    def w(*a, **k): return f"<i>{f(*a, **k)}</i>"
    return w

@bold
@italic
def text():
    return "hello"

print(text())        # <b><i>hello</i></b>

## 17.6 Memoization

**Memoization** caches a function's results so repeated calls with the same arguments are instant. It turns exponential recursion into linear time.

### By hand

In [ ]:
cache = {}

def fib(n):
    if n in cache:
        return cache[n]
    if n < 2:
        return n
    cache[n] = fib(n - 1) + fib(n - 2)
    return cache[n]

print(fib(10), fib(80))

### As a reusable decorator

In [ ]:
import functools

def memoize(func):
    store = {}
    @functools.wraps(func)
    def wrapper(*args):
        if args not in store:
            store[args] = func(*args)
        return store[args]
    wrapper.cache = store
    return wrapper

@memoize
def slow_square(n):
    print("computing", n)
    return n * n

print(slow_square(4))
print(slow_square(4))      # no "computing" line the second time

### The standard way: `functools.lru_cache`

In [ ]:
from functools import lru_cache, cache

@lru_cache(maxsize=128)             # keeps the 128 most recently used results
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(90))
print(fib.cache_info())             # hits, misses, size
fib.cache_clear()

@cache                              # Python 3.9+: unbounded cache
def square(n):
    return n * n
print(square(12))

Cached functions must be **pure** (same inputs always give the same output, no side effects) and their arguments must be **hashable**.

## 17.7 `functools.partial`

`partial` pre-fills some arguments of a function, producing a new callable.

In [ ]:
from functools import partial

def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)
cube = partial(power, exponent=3)
print(square(5), cube(2))

to_int_base2 = partial(int, base=2)
print(to_int_base2("1010"))

## 17.8 Built-in decorators you have met

| Decorator | Module |
|---|---|
| `@property`, `@classmethod`, `@staticmethod` | 14 |
| `@dataclass` | 15 |
| `@contextmanager` | 16 |
| `@lru_cache`, `@wraps` | 17 |

## Common pitfalls

- Forgetting `@wraps` (breaks introspection, help, and some frameworks).
- Writing `@repeat` when the decorator needs arguments: `@repeat(3)`.
- Caching impure functions or functions with unhashable arguments (lists, dicts).
- Unbounded caches in long-running programs — use `maxsize`.
- Stacking order confusion: the decorator nearest `def` runs first.

## Practice

1. Write a `@retry(times)` decorator that re-calls a function when it raises an exception.
2. Write a `@validate_positive` decorator that raises `ValueError` if any numeric argument is negative.
3. Compare timings of naïve `fib(32)` vs `@lru_cache` `fib(32)`.